# Projeto Fictus | Análise Financeira — Bloco 5: Cenários de Decisão e Recomendação Final

---

## Pergunta Central do Bloco
> **A saúde financeira da empresa-alvo sustenta a tese de aquisição — e qual é o custo financeiro de errar em cada direção?**

---

## Contexto do Bloco

Este bloco fecha o ciclo aberto pela Análise de Vendas. Os quatro blocos anteriores estabeleceram:

- **Bloco 1:** A estrutura de recebimento — mix de pagamento, PMR, capital em aberto e spread capturado por intermediários
- **Bloco 2:** O perfil de risco de crédito — concentração, volatilidade e impacto de choques de inadimplência
- **Bloco 3:** A rentabilidade real — spread líquido após desconto de risco, retorno ajustado ao risco e custo de oportunidade do capital
- **Bloco 4:** Capital e escalabilidade — capital imobilizado, teto de crescimento e ranking de riscos por Matriz GUT

Agora os achados dos quatro blocos são consolidados em três cenários financeiros, as premissas são testadas por sensibilidade e a recomendação final é gerada dinamicamente — verificando também se as condições estabelecidas pela Análise de Vendas foram atendidas.

---

## Os Três Cenários
| Cenário | Descrição | Questão central |
|---|---|---|
| Manter estrutura atual | Nenhuma mudança no modelo de recebimento | O modelo atual aguenta o crescimento projetado sem pressionar o caixa? |
| Otimização parcial | Renegociar condições com intermediários nas modalidades de maior spread | Captura qual % do ganho com qual nível de complexidade? |
| Reestruturação total | Revisão completa da política de recebimento e intermediação | O ganho de eficiência justifica o capital e o esforço de implementação? |

---

## Declaração de Escopo
> *Esta análise é baseada exclusivamente em dados transacionais e operacionais disponíveis no dataset Olist.*
> *A recomendação gerada considera apenas as dimensões que os dados autorizam responder.*
> *Não substitui due diligence jurídica, contábil ou regulatória.*

---


## Configuração e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_FIN     = BASE_DIR / "data" / "finance"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    if abs(x) >= 1_000:     return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.0f}"
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")

def ler(f, **kw):
    df = pd.read_csv(DIR_FIN / f, low_memory=False, **kw)
    df.columns = df.columns.str.strip()
    return df

fin_fato   = ler("fin_fato.csv")
fin_mensal = ler("fin_mensal.csv")
fin_trim   = ler("fin_trimestral.csv")
fin_pag    = ler("fin_pagamento.csv")
fin_reg    = ler("fin_regional.csv")
fin_faixa  = ler("fin_faixa.csv")

for col in ["preco", "numero_parcelas", "pmr_ajustado", "spread_intermediario",
            "capital_em_aberto", "anomalia_pagamento"]:
    if col in fin_fato.columns:
        fin_fato[col] = pd.to_numeric(fin_fato[col], errors="coerce")

# ─── Premissas centrais — auditáveis ─────────────────────────────────────────
CUSTO_OPERACIONAL_MENSAL = 25_000
CUSTO_CAPITAL_AM         = 0.0120
TAXA_INADIMPLENCIA       = fin_fato["anomalia_pagamento"].mean()
CUSTO_SETUP              = 150_000
CAPACIDADE_MAX_CARTEIRA  = 2_000_000

receita_total  = fin_fato["preco"].sum()
capital_total  = fin_fato["capital_em_aberto"].sum()
spread_total   = fin_fato["spread_intermediario"].sum()
meses_total    = fin_mensal["ano_mes"].nunique()
receita_mensal = receita_total / meses_total
capital_mensal = capital_total / meses_total
spread_mensal  = spread_total / meses_total

custo_risco_m  = receita_mensal * TAXA_INADIMPLENCIA
custo_cap_m    = capital_mensal * CUSTO_CAPITAL_AM
spread_liq_m   = spread_mensal - custo_risco_m - custo_cap_m - CUSTO_OPERACIONAL_MENSAL
pct_parcelado  = fin_fato[fin_fato["tipo_pagamento"] == "cartao_credito"]["preco"].sum() / receita_total
pmr_geral      = (fin_fato["pmr_ajustado"] * fin_fato["preco"]).sum() / receita_total
cv_risco       = fin_mensal["pct_anomalia"].std() / fin_mensal["pct_anomalia"].mean() \
                 if fin_mensal["pct_anomalia"].mean() > 0 else 0
retorno_ajust  = (spread_liq_m / (capital_mensal + CUSTO_SETUP) * 12) * (1 - cv_risco) \
                 if (capital_mensal + CUSTO_SETUP) > 0 else 0
payback        = CUSTO_SETUP / spread_liq_m if spread_liq_m > 0 else float("inf")
ratio_capital  = capital_mensal / CAPACIDADE_MAX_CARTEIRA
ponto_ruptura  = CAPACIDADE_MAX_CARTEIRA / capital_mensal if capital_mensal > 0 else None

print(f"Dados: {len(fin_fato):,} registros | {meses_total} meses")
print(f"Receita mensal    : R$ {receita_mensal:,.0f}")
print(f"Spread líq mensal : R$ {spread_liq_m:,.0f}")
print(f"Retorno ajustado  : {retorno_ajust*100:.1f}% a.a.")
print(f"Ratio capital     : {ratio_capital:.1%}")


---

## Análise 1 — Checklist: As Condições da Análise de Vendas Foram Atendidas?

> *"O veredicto da Análise de Vendas estabeleceu condições para que a aquisição fosse viável? A primeira pergunta deste bloco não é sobre crédito — é sobre se essas condições foram cumpridas com base nos dados disponíveis. A verificação dinâmica de pré-condições, derivada dos dados e não de julgamento subjetivo, garante que a análise financeira parta de uma base de entrada validada e auditável."*

**Framework:** PDCA — verificação de condições de entrada  
**Entrega:** Checklist dinâmico das condições da Análise de Vendas com status calculado pelos dados

**Como este script responde à pergunta:**
>O script percorre as condições estabelecidas no veredicto da Análise de Vendas e verifica cada uma com base nas métricas calculadas nos blocos anteriores. Cada condição recebe um status binário — atendida ou não atendida — derivado automaticamente dos dados, sem julgamento manual. Um painel responde à pergunta:  
> 1. **Checklist de condições com status:** Barras horizontais coloridas em verde (atendida) ou vermelho (não atendida), com o valor observado anotado ao lado do critério. O número de condições atendidas sobre o total define o grau de preparação do ativo para a fase financeira — e sinaliza se a análise de crédito está sendo feita sobre uma base sólida ou sobre um ativo ainda com pendências estruturais.


**Análise do Resultado:**
O checklist posiciona o ativo antes de qualquer decisão financeira. Se condições críticas da Análise de Vendas ainda não foram atendidas, as projeções financeiras dos blocos seguintes perdem sustentação — os cenários de crescimento assumem uma base que os dados ainda não confirmam. Para o comprador, condições não atendidas nesta análise são itens de negociação prioritária antes do fechamento, não problemas a resolver depois. O número de itens vermelhos define o grau de condicionalidade da recomendação final.


In [ ]:
# ─── Checklist das condições do Parte 1 ────────────────────────────────────
# As condições são verificadas com base nos dados disponíveis no Finance
# Dados adicionais do Retail são inferidos das tabelas de Finance que derivam do Retail

pct_frete = (fin_fato["valor_frete"].sum() / receita_total) * 100 if "valor_frete" in fin_fato.columns else None
nota_media = fin_fato["nota_review"].mean() if "nota_review" in fin_fato.columns else None
pct_no_prazo = fin_fato["entregue_no_prazo"].mean() * 100 if "entregue_no_prazo" in fin_fato.columns else None

conditions_p1 = [
    {
        "condicao"   : "Frete abaixo de 20% da receita",
        "metrica"    : f"{pct_frete:.1f}%" if pct_frete is not None else "N/D",
        "status"     : pct_frete < 20 if pct_frete is not None else None,
        "threshold"  : "< 20%",
        "fonte"      : "Derivado dos dados do Retail via Finance ETL",
    },
    {
        "condicao"   : "SLA (% entregas no prazo) acima de 90%",
        "metrica"    : f"{pct_no_prazo:.1f}%" if pct_no_prazo is not None else "N/D",
        "status"     : pct_no_prazo >= 90 if pct_no_prazo is not None else None,
        "threshold"  : "≥ 90%",
        "fonte"      : "Derivado dos dados do Retail via Finance ETL",
    },
    {
        "condicao"   : "Nota média de review acima de 4.0",
        "metrica"    : f"{nota_media:.2f}" if nota_media is not None else "N/D",
        "status"     : nota_media >= 4.0 if nota_media is not None else None,
        "threshold"  : "≥ 4.0",
        "fonte"      : "Derivado dos dados do Retail via Finance ETL",
    },
    {
        "condicao"   : "Taxa de anomalia de pagamento < 15%",
        "metrica"    : f"{TAXA_INADIMPLENCIA*100:.2f}%",
        "status"     : TAXA_INADIMPLENCIA < 0.15,
        "threshold"  : "< 15%",
        "fonte"      : "Calculado no Finance — Bloco 2",
    },
    {
        "condicao"   : "Spread líquido mensal positivo",
        "metrica"    : f"R$ {spread_liq_m:,.0f}/mês",
        "status"     : spread_liq_m > 0,
        "threshold"  : "> R$ 0",
        "fonte"      : "Calculado no Finance — Bloco 3",
    },
]

print("=" * 65)
print("CHECKLIST — CONDIÇÕES PRÉ-DECISÃO DE CRÉDITO")
print("=" * 65)
for c in conditions_p1:
    if c["status"] is None:
        icone = "❓"
    elif c["status"]:
        icone = "✅"
    else:
        icone = "❌"
    print(f"  {icone} {c['condicao']:<45} | Atual: {c['metrica']:<10} | Meta: {c['threshold']}")

n_ok  = sum(1 for c in conditions_p1 if c["status"] is True)
n_nao = sum(1 for c in conditions_p1 if c["status"] is False)
print(f"\n  {n_ok}/{len(conditions_p1)} condições atendidas")
if n_nao > 0:
    print(f"  ⚠️  {n_nao} condição(ões) não atendida(s) — revisar antes de internalizar")


---

## Análise 2 — Comparação dos Três Cenários

> *"Os dados dos quatro blocos anteriores convergem aqui para uma comparação direta: qual estrutura financeira produz o melhor resultado para o comprador nas condições específicas da empresa-alvo? O Planejamento por Cenários traduz os achados analíticos em números comparáveis — resultado líquido, capital necessário e risco herdado — permitindo que a decisão seja tomada com base em evidência, não em preferência."*

**Framework:** Planejamento por Cenários  
**Entrega:** Matriz de condições de superioridade por cenário com fronteiras numéricas

**Como este script responde à pergunta:**
>O script calcula o resultado financeiro de 12 meses para os três cenários — manter terceirizado, internalizar parcialmente e internalizar totalmente — e os compara em quatro dimensões: spread líquido mensal, capital necessário, risco herdado e payback. Um painel responde à pergunta:  
> 1. **Barras comparativas por dimensão:** Cada grupo de barras representa uma dimensão financeira, com os três cenários lado a lado. A barra mais favorável em cada dimensão é destacada. O score total de cada cenário — soma das dimensões onde ele é superior — aparece no topo do gráfico, permitindo identificar qual cenário vence mais dimensões simultaneamente e qual envolve maiores trade-offs entre capital e retorno.


**Análise do Resultado:**
A comparação por dimensões evita a armadilha de escolher o cenário com maior resultado nominal sem considerar o capital necessário e o risco assumido. O cenário vencedor não é necessariamente o de maior spread líquido — é o que equilibra retorno, capital e risco dentro das condições específicas do ativo. Se nenhum cenário domina em todas as dimensões, os trade-offs explícitos orientam a negociação: o comprador pode aceitar menor retorno em troca de menor capital imobilizado, ou vice-versa, a depender da sua estrutura de funding e apetite de risco.

In [ ]:
# ─── Modelagem dos três cenários ──────────────────────────────────────────────
# Cenário 1: Manter terceirizado
c1_spread_mensal     = 0  # spread fica com intermediários
c1_custo_mensal      = 0  # sem custo operacional de crédito
c1_capital_mensal    = 0  # sem capital imobilizado
c1_risco_crescimento = receita_mensal * TAXA_INADIMPLENCIA * 0.3  # impacto residual
c1_resultado_mensal  = c1_spread_mensal - c1_risco_crescimento
c1_capital_inicial   = 0

# Cenário 2: Modelo híbrido (apenas cartão de crédito parcelado > 3x, top regiões)
# Captura ~60% do spread com ~40% do capital e risco
fator_hibrido        = 0.60
c2_spread_mensal     = spread_mensal * fator_hibrido
c2_custo_mensal      = CUSTO_OPERACIONAL_MENSAL * 0.75  # equipe menor
c2_capital_mensal    = capital_mensal * fator_hibrido
c2_risco_mensal      = receita_mensal * fator_hibrido * TAXA_INADIMPLENCIA
c2_custo_cap         = c2_capital_mensal * CUSTO_CAPITAL_AM
c2_resultado_mensal  = c2_spread_mensal - c2_custo_mensal - c2_risco_mensal - c2_custo_cap
c2_capital_inicial   = CUSTO_SETUP * 0.7

# Cenário 3: Internalizar totalmente
c3_spread_mensal     = spread_mensal
c3_custo_mensal      = CUSTO_OPERACIONAL_MENSAL
c3_capital_mensal    = capital_mensal
c3_risco_mensal      = custo_risco_m
c3_custo_cap         = custo_cap_m
c3_resultado_mensal  = spread_liq_m
c3_capital_inicial   = CUSTO_SETUP

cenarios = [
    {"nome": "Manter Terceirizado",   "resultado_m": c1_resultado_mensal,
     "capital_inicial": c1_capital_inicial, "capital_m": c1_capital_mensal,
     "pct_spread": 0,       "cor": COR_NEUTRO},
    {"nome": "Modelo Híbrido",         "resultado_m": c2_resultado_mensal,
     "capital_inicial": c2_capital_inicial, "capital_m": c2_capital_mensal,
     "pct_spread": fator_hibrido*100,  "cor": COR_RECEITA},
    {"nome": "Internalizar Totalmente","resultado_m": c3_resultado_mensal,
     "capital_inicial": c3_capital_inicial, "capital_m": c3_capital_mensal,
     "pct_spread": 100,     "cor": COR_MARGEM if c3_resultado_mensal > 0 else COR_ALERTA},
]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle("Bloco 5 — Comparação dos Três Cenários de Decisão",
             fontsize=13, fontweight="bold")

# Painel 1: Resultado líquido mensal
nomes = [c["nome"] for c in cenarios]
resultados = [c["resultado_m"] for c in cenarios]
cores_cen  = [c["cor"] for c in cenarios]
bars1 = axes[0].bar(nomes, [r / 1000 for r in resultados], color=cores_cen, alpha=0.85)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Resultado Líquido Mensal Estimado (R$ mil)", fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[0].tick_params(axis="x", rotation=20)
for bar, val in zip(bars1, resultados):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.2 if val >= 0 else -2),
                 f"R${val/1000:+,.0f}K", ha="center", va="bottom", fontsize=9, fontweight="bold")

# Painel 2: Capital imobilizado
capitais = [c["capital_inicial"] for c in cenarios]
bars2 = axes[1].bar(nomes, [cap / 1000 for cap in capitais], color=cores_cen, alpha=0.85)
axes[1].set_title("Capital de Setup Necessário (R$ mil)", fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[1].tick_params(axis="x", rotation=20)
for bar, val in zip(bars2, capitais):
    if val > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f"R${val/1000:,.0f}K", ha="center", va="bottom", fontsize=9)
    else:
        axes[1].text(bar.get_x() + bar.get_width()/2, 1,
                     "Sem investimento", ha="center", va="bottom", fontsize=8, color=COR_NEUTRO)

# Painel 3: % do spread capturado vs. resultado líquido
axes[2].scatter([c["pct_spread"] for c in cenarios],
                [c["resultado_m"] / 1000 for c in cenarios],
                c=[c["cor"] for c in cenarios], s=300, alpha=0.9, zorder=5)
for c in cenarios:
    axes[2].annotate(c["nome"],
                     (c["pct_spread"], c["resultado_m"] / 1000),
                     textcoords="offset points", xytext=(8, 5), fontsize=8)
axes[2].axhline(0, color=COR_ALERTA, linestyle="--", linewidth=1, alpha=0.6)
axes[2].set_title("% Spread Capturado vs. Resultado Líquido (R$ mil)", fontsize=9)
axes[2].set_xlabel("% do Spread Total Capturado")
axes[2].set_ylabel("Resultado Líquido Mensal (R$ mil)")
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))

plt.tight_layout()
salvar(fig, "05_comparacao_cenarios")
plt.show()


---

## Análise 3 — Custo de Errar em Cada Direção

> *"A decisão de reestruturar o modelo de recebimento é assimétrica: os custos de errar em cada direção são diferentes em natureza e magnitude. Agir antes de atingir as condições ideais imobiliza capital sem retorno proporcional e pode comprometer o caixa operacional. Aguardar além do ponto ótimo mantém a erosão de margem pelos intermediários a cada trimestre de inação. A comparação dos dois custos ao longo do tempo revela o momento financeiramente fundamentado para agir — e a assimetria real entre os dois tipos de erro."*

**Framework:** Custo de Oportunidade + Análise de Sensibilidade  
**Entrega:** Comparação assimétrica do custo de antecipação versus custo de postergação

**Como este script responde à pergunta:**
>O script calcula, trimestre a trimestre por dois anos, o custo acumulado de antecipar a decisão e o custo acumulado de postergar. O ponto onde as duas curvas se cruzam é o momento ótimo — antes dele, esperar custa menos; depois dele, agir custa menos. Um painel responde à pergunta:  
> 1. **Curvas de custo acumulado por trimestre:** Linha azul para o custo de antecipação e linha vermelha para o custo de postergação, com área sombreada entre as curvas representando o diferencial acumulado. O cruzamento é marcado com anotação do trimestre ótimo. A inclinação relativa das duas curvas revela qual tipo de erro é mais caro: se a curva de antecipação sobe mais rápido, o risco de agir cedo é maior; se a de postergação é mais íngreme, a urgência de agir é imediata.


**Análise do Resultado:**
O trimestre de cruzamento das curvas é o dado mais acionável desta análise: define a janela de decisão financeiramente fundamentada. Antes desse ponto, o comprador tem evidência de que aguardar o volume atingir o break-even é a decisão mais eficiente em termos de capital. Depois dele, cada trimestre de inação tem custo crescente e mensurável — o argumento financeiro direto para acelerar a decisão. A assimetria entre os dois custos também informa a estratégia de negociação: se o custo de antecipação é muito maior que o de postergação, o comprador tem margem para negociar um prazo de implementação mais conservador sem comprometer o retorno esperado.

In [ ]:
# ─── Custo de errar: antecipação vs. postergação ──────────────────────────────
trimestres = list(range(1, 9))  # 8 trimestres = 2 anos

# Custo de antecipar errado: capital imobilizado sem retorno proporcional
# Cenário pessimista: spread líquido 50% menor que o projetado
spread_pessimista = spread_liq_m * 0.5
custo_antecipacao = [
    max(0, CUSTO_SETUP + capital_mensal * 3 * t - spread_pessimista * 3 * t) / 1000
    for t in trimestres
]

# Custo de postergar: spread deixado na mesa por trimestre
custo_postergacao = [
    spread_liq_m * 3 * t / 1000 if spread_liq_m > 0 else 0
    for t in trimestres
]

fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle("Bloco 5 — Assimetria: Custo de Antecipar vs. Custo de Postergar (R$ mil)",
             fontsize=13, fontweight="bold")

ax.plot(trimestres, custo_antecipacao, color=COR_ALERTA, linewidth=2.5,
        marker="o", markersize=7, label="Custo de antecipar (cenário pessimista)")
ax.plot(trimestres, custo_postergacao, color=COR_RECEITA, linewidth=2.5,
        marker="s", markersize=7, label="Custo de postergar (oportunidade perdida)")
ax.fill_between(trimestres, custo_antecipacao, custo_postergacao,
                where=[a > p for a, p in zip(custo_antecipacao, custo_postergacao)],
                alpha=0.1, color=COR_ALERTA, label="Zona: antecipar > postergar")
ax.fill_between(trimestres, custo_antecipacao, custo_postergacao,
                where=[p >= a for a, p in zip(custo_antecipacao, custo_postergacao)],
                alpha=0.1, color=COR_RECEITA, label="Zona: postergar > antecipar")
ax.set_xticks(trimestres)
ax.set_xticklabels([f"T{t}" for t in trimestres])
ax.set_xlabel("Trimestres")
ax.set_ylabel("Custo Acumulado (R$ mil)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
ax.legend(fontsize=9)

plt.tight_layout()
salvar(fig, "05_custo_assimetria")
plt.show()

# Ponto de cruzamento (onde os custos se igualam)
for t in trimestres:
    if custo_postergacao[t-1] >= custo_antecipacao[t-1]:
        print(f"\n  Ponto de cruzamento: T{t} ({t*3} meses)")
        print(f"  A partir desse trimestre, o custo de postergar supera o de antecipar.")
        break
else:
    print(f"\n  Custo de antecipar supera o de postergar em todo o horizonte analisado.")
    print(f"  Recomendação: aguardar condições mais favoráveis antes de internalizar.")


---

## Análise 4 — Recomendação Final: Os Três Projetos Formam uma Estratégia Coerente?

> *"A recomendação financeira é a última peça da due diligence completa. Este bloco responde não apenas se a estrutura financeira da empresa-alvo é saudável — mas se o conjunto das três frentes de análise — Vendas, Logística e Finanças — converge para uma tese de aquisição coerente e sustentável. Uma frente favorável isolada pode ser insuficiente se as demais apontam riscos estruturais não endereçados."*

**Framework:** Matriz de Decisão consolidada — integração das três frentes analíticas  
**Entrega:** Score consolidado dos quatro blocos financeiros com recomendação derivada dos dados e checagem de coerência estratégica entre as três frentes

**Como este script responde à pergunta:**
>O script consolida os scores dos quatro blocos financeiros, integra com os sinais da Análise de Vendas e da Análise Logística, e verifica se as três recomendações apontam na mesma direção. Dois painéis respondem à pergunta:  
> 1. **Score por bloco financeiro:** Barras horizontais com o score de cada bloco (1 = risco alto, 2 = atenção, 3 = positivo), com o sinal textual anotado ao lado. O score médio e o número de blocos em cada nível definem o perfil financeiro consolidado do ativo.
> 2. **Coerência estratégica entre as três frentes:** Painel comparativo mostrando a recomendação das três frentes lado a lado — Vendas, Logística e Finanças — com indicação se apontam na mesma direção. Três recomendações convergentes fortalecem a tese; divergências sinalizam que a estratégia de integração precisa ser revisada antes do fechamento.


**Análise do Resultado:**
A coerência entre as três frentes é o teste final da tese de aquisição. Uma recomendação positiva em Vendas combinada com restrições financeiras severas indica que o ativo tem potencial comercial mas estrutura de capital inadequada — o comprador precisaria aportar capital antes de capturar valor. Uma recomendação financeira positiva com restrições logísticas indica que a eficiência financeira depende de uma intervenção operacional que ainda não foi executada. Apenas quando as três frentes convergem para a mesma direção a tese de aquisição é robusta o suficiente para avançar sem condicionantes estruturais.

In [ ]:
# ─── Scores consolidados dos 4 blocos do Finance ─────────────────────────────
# PMR e concentração em parcelado
_pct_parcelado    = pct_parcelado * 100
_pmr_geral        = pmr_geral
_b1_score = 1 if (_pct_parcelado > 60 and _pmr_geral > 45) else 2 if (_pct_parcelado > 60 or _pmr_geral > 45) else 3
_b1_cor   = COR_ALERTA if _b1_score == 1 else COR_DESTAQUE if _b1_score == 2 else COR_MARGEM
_b1_sinal = ("⚠️  Estrutura de recebimento pressionada" if _b1_score == 1
             else "⚠️  PMR elevado e/ou alta concentração em parcelado" if _b1_score == 2
             else "✅ Estrutura de recebimento equilibrada")

# Risco e variabilidade
_taxa_anom = TAXA_INADIMPLENCIA * 100
_b2_score = 1 if (_taxa_anom > 15 and cv_risco > 0.3) else 2 if (_taxa_anom > 15 or cv_risco > 0.3) else 3
_b2_cor   = COR_ALERTA if _b2_score == 1 else COR_DESTAQUE if _b2_score == 2 else COR_MARGEM
_b2_sinal = ("🔴 Risco elevado e volátil" if _b2_score == 1
             else "⚠️  Risco moderado — monitoramento rigoroso" if _b2_score == 2
             else "✅ Risco controlado e previsível")

# Rentabilidade
_b3_score = 1 if spread_liq_m <= 0 else 2 if retorno_ajust < 0.10 else 3
_b3_cor   = COR_ALERTA if _b3_score == 1 else COR_DESTAQUE if _b3_score == 2 else COR_MARGEM
_b3_sinal = ("🔴 Spread líquido negativo" if _b3_score == 1
             else "⚠️  Retorno marginal abaixo do custo de capital" if _b3_score == 2
             else f"✅ Spread positivo e retorno de {retorno_ajust*100:.1f}% a.a.")

# Capital e escala
_b4_score = 1 if ratio_capital > 0.9 else 2 if (ponto_ruptura is not None and ponto_ruptura < 2.0) else 3
_b4_cor   = COR_ALERTA if _b4_score == 1 else COR_DESTAQUE if _b4_score == 2 else COR_MARGEM
_b4_sinal = ("🔴 Capital já próximo do limite máximo" if _b4_score == 1
             else f"⚠️  Teto de crescimento em ×{ponto_ruptura:.1f}" if _b4_score == 2
             else "✅ Capital disponível suporta crescimento projetado")

blocos = [
    {"bloco": "Bloco 1 — Perfil de Pagamento",     "score": _b1_score, "cor": _b1_cor,
     "pergunta": "O modelo sustenta o crescimento?",
     "achado": f"PMR médio: {_pmr_geral:.0f}d | {_pct_parcelado:.1f}% parcelado",
     "sinal": _b1_sinal,
     "cond": f"Reduzir PMR e avaliar mix antes de internalizar." if _b1_score <= 2
             else "Estrutura favorável à internalização."},
    {"bloco": "Bloco 2 — Risco de Crédito",         "score": _b2_score, "cor": _b2_cor,
     "pergunta": "O risco é internalizável?",
     "achado": f"Taxa de anomalia: {_taxa_anom:.2f}% | CV: {cv_risco:.2f}",
     "sinal": _b2_sinal,
     "cond": f"Capital de reserva ≥ {_taxa_anom*2:.1f}% da receita mensal." if _b2_score <= 2
             else "Perfil de risco compatível com avaliação."},
    {"bloco": "Bloco 3 — Rentabilidade",             "score": _b3_score, "cor": _b3_cor,
     "pergunta": "O spread compensa o risco?",
     "achado": f"Spread líq: R$ {spread_liq_m:,.0f}/mês | Retorno: {retorno_ajust*100:.1f}% a.a.",
     "sinal": _b3_sinal,
     "cond": "Revisitar custo operacional e custo de capital." if _b3_score <= 2
             else "Rentabilidade sustenta internalização parcial."},
    {"bloco": "Bloco 4 — Capital e Escala",          "score": _b4_score, "cor": _b4_cor,
     "pergunta": "A holding suporta o capital necessário?",
     "achado": f"Capital/Capacidade: {ratio_capital:.1%} | Teto: ×{ponto_ruptura:.1f}" if ponto_ruptura else f"Capital: {ratio_capital:.1%}",
     "sinal": _b4_sinal,
     "cond": f"Definir política de carteira máxima antes de expandir." if _b4_score <= 2
             else "Estrutura de capital compatível com internalização gradual."},
]

# ─── Gráfico de consolidação ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("RELATÓRIO FINAL — Consolidação dos 4 Blocos | FICTUS | Análise Financeira",
             fontsize=14, fontweight="bold")

nomes_blocos  = [b["bloco"].split(" — ")[1] for b in blocos]
scores_blocos = [b["score"] for b in blocos]
cores_blocos  = [b["cor"] for b in blocos]

bars = axes[0].barh(nomes_blocos, scores_blocos, color=cores_blocos, alpha=0.85)
axes[0].set_xlim(0, 3.5)
axes[0].set_xticks([1, 2, 3])
axes[0].set_xticklabels(["Risco Alto", "Atenção", "Positivo"], fontsize=9)
axes[0].axvline(1.5, color=COR_NEUTRO, linewidth=0.5, alpha=0.4)
axes[0].axvline(2.5, color=COR_NEUTRO, linewidth=0.5, alpha=0.4)
for bar, b in zip(bars, blocos):
    axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 b["sinal"][:50], va="center", fontsize=7.5, color="#333333")
axes[0].set_title("Score por Bloco Decisório", fontsize=11)

# Comparação dos cenários
nomes_cen = [c["nome"] for c in cenarios]
result_cen = [c["resultado_m"] for c in cenarios]
cores_cen2 = [COR_NEUTRO, COR_RECEITA, COR_MARGEM if c3_resultado_mensal > 0 else COR_ALERTA]
axes[1].bar(nomes_cen, [r / 1000 for r in result_cen], color=cores_cen2, alpha=0.85)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Resultado Líquido Mensal por Cenário (R$ mil)", fontsize=11)
axes[1].set_ylabel("Resultado (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[1].tick_params(axis="x", rotation=15)
for i, (nome, val) in enumerate(zip(nomes_cen, result_cen)):
    axes[1].text(i, (val / 1000) + (0.3 if val >= 0 else -2),
                 f"R${val/1000:+,.0f}K", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
salvar(fig, "05_relatorio_consolidado_finance")
plt.show()


In [ ]:
# ─── Recomendação final — 100% derivada dos dados ────────────────────────────
score_medio = sum(b["score"] for b in blocos) / len(blocos)
n_score1    = sum(1 for b in blocos if b["score"] == 1)
n_score2    = sum(1 for b in blocos if b["score"] == 2)
n_score3    = sum(1 for b in blocos if b["score"] == 3)

# Cenário recomendado derivado dos scores
if n_score1 >= 2 or spread_liq_m <= 0:
    decisao_credito      = "🔴 NÃO INTERNALIZAR — riscos e/ou spread não sustentam a decisão"
    decisao_curta        = "NÃO INTERNALIZAR"
    cenario_recomendado  = "Manter Terceirizado"
    emoji_decisao        = "🔴"
elif n_score1 == 1 or (n_score2 >= 2 and score_medio < 2.2):
    decisao_credito      = "⚠️  INTERNALIZAÇÃO HÍBRIDA — modalidades de baixo risco apenas"
    decisao_curta        = "MODELO HÍBRIDO"
    cenario_recomendado  = "Modelo Híbrido"
    emoji_decisao        = "⚠️ "
else:
    decisao_credito      = "✅ INTERNALIZAÇÃO GRADUAL — condições favorecem expansão controlada"
    decisao_curta        = "INTERNALIZAÇÃO GRADUAL"
    cenario_recomendado  = "Modelo Híbrido → Total"
    emoji_decisao        = "✅"

condicoes = [b["cond"] for b in blocos if b["score"] <= 2]

# ─── Impressão do relatório ──────────────────────────────────────────────────
print("=" * 72)
print("RECOMENDAÇÃO FINAL — FICTUS | Análise Financeira")
print("=" * 72)
print(f"""
  Decisão recomendada: {decisao_credito}
  Cenário sugerido   : {cenario_recomendado}
""")

print("─" * 72)
print("PARA O BOARD — EM LINGUAGEM DE DECISÃO")
print("─" * 72)
print(f"""
O QUE OS DADOS MOSTRAM

A estrutura de pagamentos da FICTUS é dominada por cartão de crédito
parcelado, com PMR médio de {_pmr_geral:.0f} dias. Isso significa que parcela
significativa da receita já está comprometida com intermediários financeiros,
que capturam um spread estimado de R$ {spread_mensal:,.0f}/mês da operação.

O risco de crédito latente (taxa de anomalia: {_taxa_anom:.2f}%) é {'elevado e' if _taxa_anom > 10 else ''}{'volátil' if cv_risco > 0.3 else 'relativamente controlado'},
o que {'dificulta' if cv_risco > 0.3 else 'não impede'} a precificação interna confiável.

O spread líquido após desconto de risco, custo de capital e operação é
de R$ {spread_liq_m:,.0f}/mês — {'positivo, mas' if spread_liq_m > 0 else 'negativo, o que'} {'requer avaliação cuidadosa do cenário híbrido.' if spread_liq_m > 0 else 'inviabiliza a internalização total nas premissas atuais.'}
""")

print("─" * 72)
print("O QUE A INTERNALIZAÇÃO COMPRARIA")
print("─" * 72)
print(f"""
  Benefício potencial:
  • Spread capturado internamente  : R$ {spread_total:,.0f} no período
  • Spread médio mensal potencial  : R$ {spread_mensal:,.0f}/mês
  • Controle sobre concessão de crédito e parcelamento

  Custos e riscos herdados:
  • Capital imobilizado médio       : R$ {capital_mensal:,.0f}/mês
  • Custo operacional de crédito    : R$ {CUSTO_OPERACIONAL_MENSAL:,.0f}/mês
  • Taxa de anomalia atual (proxy)  : {_taxa_anom:.2f}% — requer reserva de capital
  • Teto de crescimento             : ×{ponto_ruptura:.1f} o volume atual sem funding adicional
""")

if condicoes:
    print("─" * 72)
    print(f"AS {len(condicoes)} CONDIÇÕES PARA A INTERNALIZAÇÃO SER BEM-SUCEDIDA")
    print("─" * 72)
    for i, cond in enumerate(condicoes, 1):
        print(f"  {i}. {cond}")

print(f"""
  + Recomendação de estruturação:
    {'Iniciar pelo Modelo Híbrido: internalizar apenas cartão parcelado > 3x em SP/RJ/MG,' if cenario_recomendado != 'Manter Terceirizado' else 'Manter o modelo atual e monitorar spread trimestral.'}
    {'com capital inicial de R$ ' + f"{c2_capital_inicial:,.0f}" + ' e avaliação de expansão em 12 meses.' if cenario_recomendado != 'Manter Terceirizado' else ''}
""")

print("─" * 72)
print("SÍNTESE DA FICTUS CASE SERIES — AS TRÊS DECISÕES")
print("─" * 72)
print("""
  Parte 1 — FICTUS | Análise de Vendas    : Aquisição (condicional ou recomendada)
  Parte 2 — FICTUS | Análise Logística : Internalização logística (híbrida ou total)
  Parte 3 — FICTUS | Análise Financeira   : Internalização de crédito (avaliação acima)

  As três decisões são interdependentes. A internalização de crédito só faz
  sentido após a logística estar estabilizada — caixa pressionado por SLA
  ruim e por carteira de crédito imobilizada é o caminho mais curto para
  uma crise de liquidez operacional.
""")

print("=" * 72)
print("SUMÁRIO EXECUTIVO — FICTUS | Análise Financeira")
print("=" * 72)
for b in blocos:
    nivel = "✅" if b["score"] == 3 else "⚠️ " if b["score"] == 2 else "🔴"
    nome  = b["bloco"].split(" — ")[1]
    print(f"  {nivel}  {nome:<35} {b['sinal']}")
print(f"  {emoji_decisao}  {'Recomendação Final':<35} {decisao_curta}")
print("=" * 72)
print("""
  DECLARAÇÃO DE ESCOPO:
  Esta análise é baseada exclusivamente em dados transacionais e operacionais disponíveis no dataset. A recomendação considera apenas as dimensões que os dados autorizam a responder. Não substitui due diligence jurídica, contábil
  ou regulatória — especialmente no que tange a licenciamento para operação de crédito e regulamentação do Banco Central do Brasil.

  Parte 4 — Executive AI Report Layer disponível: cole o Summary Log abaixo na IA de sua escolha para obter o parecer executivo consolidado da série.
""")


---
## Síntese do Bloco 5 — Cenários e Recomendação Final

> **Limitações desta análise:** todas as premissas financeiras — custo operacional mensal, custo de capital, capacidade máxima de carteira e custo de setup — são estimativas de benchmark declaradas no código e não cotações reais. A taxa de inadimplência é calculada a partir de anomalias de pagamento no dataset Olist, que é um proxy analítico e não equivale à inadimplência real de uma carteira de crédito. O retorno ajustado ao risco usa coeficiente de variação como penalizador de volatilidade — uma simplificação que subestima riscos de cauda. A recomendação final não substitui due diligence contábil, jurídica ou regulatória — especialmente no que se refere a contratos de intermediação financeira vigentes e suas condições de renegociação.


---

*Este notebook encerra a **Fase 3 — Finanças** do Projeto Fictus.*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.


---

## Summary Log — Input para a Parte 4 (Executive AI Report)

> *Cole o output desta célula no Prompt Mestre da Parte 4 para gerar o parecer executivo consolidado da FICTUS | Série de Análise.*

In [ ]:
import io as _io, sys as _sys
from datetime import datetime as _dt

def salvar_summary_log(conteudo: str, frente: str) -> None:
    """Salva o Summary Log em summary_logs/ com timestamp."""
    pasta = BASE_DIR / 'summary_logs'
    pasta.mkdir(parents=True, exist_ok=True)
    timestamp = _dt.now().strftime('%Y%m%d_%H%M')
    nome = f'summary_log_{frente}_{timestamp}.txt'
    (pasta / nome).write_text(conteudo, encoding='utf-8')
    print(f'✅ Summary Log salvo em: summary_logs/{nome}')

# Captura output e salva em summary_logs/
_buf = _io.StringIO()
_orig = _sys.stdout
_sys.stdout = _buf
try:
    # ─── Summary Log — Ficha Técnica para o Prompt Mestre ────────────────────────
    print("=" * 72)
    print("FICTUS | Análise Financeira — SUMMARY LOG (copiar para o Prompt Mestre)")
    print("=" * 72)
    print(f"""
    [FICTUS | Análise Financeira — DADOS CALCULADOS]
    
    Período analisado         : {fin_mensal['ano_mes'].min()} a {fin_mensal['ano_mes'].max()}
    Total de pedidos          : {len(fin_fato):,}
    Receita total do período  : R$ {receita_total:,.0f}
    Receita média mensal      : R$ {receita_mensal:,.0f}
    
    --- PERFIL DE PAGAMENTO ---
    Participação cartão crédito: {_pct_parcelado:.1f}% da receita
    PMR médio ponderado        : {_pmr_geral:.0f} dias
    Spread capturado por intermed: R$ {spread_total:,.0f} no período (R$ {spread_mensal:,.0f}/mês)
    Capital em aberto médio    : R$ {capital_mensal:,.0f}/mês
    
    --- RISCO DE CRÉDITO (PROXY) ---
    Taxa de anomalia geral     : {_taxa_anom:.2f}%
    Coeficiente de variação CV : {cv_risco:.2f}
    Concentração de risco      : {'Alta — SP+RJ+MG dominam' if fin_reg.head(3)['pct_acum'].iloc[-1] > 70 else 'Moderada'}
    
    --- RENTABILIDADE ---
    Spread bruto mensal        : R$ {spread_mensal:,.0f}
    Spread líquido mensal      : R$ {spread_liq_m:,.0f}
    Retorno ajustado ao risco  : {retorno_ajust*100:.1f}% a.a.
    Payback do investimento    : {f'{payback:.1f} meses' if spread_liq_m > 0 else 'indefinido (spread negativo)'}
    
    --- CAPITAL E ESCALA ---
    Capital/Capacidade máxima  : {ratio_capital:.1%}
    Teto de crescimento        : ×{ponto_ruptura:.1f} volume atual (sem funding externo)
    Custo de setup estimado    : R$ {CUSTO_SETUP:,.0f}
    
    --- SCORES POR BLOCO ---
    Bloco 1 — Perfil Pagamento : {_b1_score}/3 — {_b1_sinal}
    Bloco 2 — Risco de Crédito : {_b2_score}/3 — {_b2_sinal}
    Bloco 3 — Rentabilidade    : {_b3_score}/3 — {_b3_sinal}
    Bloco 4 — Capital/Escala   : {_b4_score}/3 — {_b4_sinal}
    Score médio                : {score_medio:.1f}/3
    
    --- RECOMENDAÇÃO ---
    Cenário recomendado        : {cenario_recomendado}
    Decisão                    : {decisao_curta}
    
    --- DECLARAÇÃO DE ESCOPO ---
    Análise baseada em dados transacionais (Olist dataset).
    Não inclui: jurídico, contábil, regulatório BACEN, gestão/fundadores.
    Variáveis de risco são proxies analíticos — não inadimplência real.
    """)
    print("=" * 72)
    
finally:
    _sys.stdout = _orig
    _conteudo = _buf.getvalue()
    print(_conteudo)
    salvar_summary_log(_conteudo, 'financas')
